# Student Performance Analysis

**Analyzing how demographics, parental education, and test preparation affect student exam scores.**

## Project Overview

This project explores the Students Performance in Exams dataset to understand how gender, ethnicity, parental education, lunch program participation, and test preparation courses affect math, reading, and writing scores.

## Learning Objectives

1. Analyze educational performance data with multiple categorical factors
2. Compare score distributions across demographic groups
3. Assess the impact of interventions (test prep courses)
4. Apply ANOVA and t-tests for group comparisons
5. Identify achievement gaps and their correlates

## Business / Research Problem

Understanding achievement gaps helps educators and policymakers:
- **Schools** allocate resources effectively
- **Policymakers** design equitable education programs
- **Parents** understand factors affecting performance

**Key question:** *Which factors most significantly impact student exam performance, and how large are the effects?*

## Why This Analysis Matters

Education equity is a critical social issue. Understanding performance drivers helps close achievement gaps and improve outcomes for all students.

## Dataset Overview

| Feature | Description |
|---------|-------------|
| `gender` | Male/Female |
| `race/ethnicity` | Group A through E |
| `parental level of education` | Education level |
| `lunch` | Standard or free/reduced |
| `test preparation course` | Completed or none |
| `math score` | 0-100 |
| `reading score` | 0-100 |
| `writing score` | 0-100 |

**Size:** 1,000 students

## Dataset Source & License Notes

- **Source:** [Kaggle - Students Performance](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)
- **License:** CC0 Public Domain
- **Note:** Synthetic/anonymized data

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'spscientist/students-performance-in-exams'
SIGNIFICANCE_LEVEL = 0.05
SCORE_COLS = ['math score', 'reading score', 'writing score']
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'Missing: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nColumns: {list(df.columns)}')
df.describe()

## Data Cleaning

In [ ]:
df = df.drop_duplicates()

# Adjust column names if needed
score_cols = [c for c in df.columns if 'score' in c.lower()]
print(f'Score columns: {score_cols}')

# Create total and average score
df['total_score'] = df[score_cols].sum(axis=1)
df['avg_score'] = df[score_cols].mean(axis=1).round(1)

# Create pass/fail based on 50 threshold
df['all_pass'] = (df[score_cols] >= 50).all(axis=1)
print(f'Pass rate (all subjects >= 50): {df["all_pass"].mean()*100:.1f}%')
print(f'Average total score: {df["total_score"].mean():.1f}/300')

## Exploratory Data Analysis

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f'\n{col}: {df[col].value_counts().to_dict()}')

print(f'\n=== Score Means ===')
for sc in score_cols:
    print(f'{sc}: {df[sc].mean():.1f}')

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'coral', 'mediumseagreen']
for i, sc in enumerate(score_cols):
    axes[i].hist(df[sc], bins=20, color=colors[i], edgecolor='white')
    axes[i].axvline(df[sc].mean(), color='red', linestyle='--', label=f'Mean: {df[sc].mean():.0f}')
    axes[i].set_title(f'{sc.title()} Distribution')
    axes[i].legend()
plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [ ]:
gender_col = [c for c in df.columns if 'gender' in c.lower()][0]
lunch_col = [c for c in df.columns if 'lunch' in c.lower()][0]
prep_col = [c for c in df.columns if 'prep' in c.lower() or 'test' in c.lower()][0]
parent_col = [c for c in df.columns if 'parent' in c.lower()][0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x=gender_col, y='avg_score', ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Average Score by Gender')

sns.boxplot(data=df, x=lunch_col, y='avg_score', ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Average Score by Lunch Type')

sns.boxplot(data=df, x=prep_col, y='avg_score', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Average Score by Test Prep')

race_col = [c for c in df.columns if 'race' in c.lower() or 'ethnicity' in c.lower()]
if race_col:
    sns.boxplot(data=df, x=race_col[0], y='avg_score', ax=axes[1,1], palette='viridis')
    axes[1,1].set_title('Average Score by Race/Ethnicity')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [ ]:
# Impact of test preparation
print('=== Test Preparation Impact ===')
for sc in score_cols:
    prep_means = df.groupby(prep_col)[sc].mean()
    print(f'{sc}: {prep_means.to_dict()}')

print(f'\n=== Parental Education Impact ===')
parent_scores = df.groupby(parent_col)['avg_score'].mean().sort_values(ascending=False)
print(parent_scores.round(1))

## Statistical Checks / Hypothesis Testing

In [ ]:
# T-test: Test prep impact
for sc in score_cols:
    completed = df[df[prep_col] == 'completed'][sc]
    none = df[df[prep_col] == 'none'][sc]
    t_stat, p_val = stats.ttest_ind(completed, none)
    print(f'{sc}: t={t_stat:.2f}, p={p_val:.4f} ({"Significant" if p_val < SIGNIFICANCE_LEVEL else "Not sig."})')

# T-test: Lunch type impact
print(f'\n=== Lunch Type ===')
std = df[df[lunch_col] == 'standard']['avg_score']
free = df[df[lunch_col] != 'standard']['avg_score']
t_stat, p_val = stats.ttest_ind(std, free)
print(f'avg_score: t={t_stat:.2f}, p={p_val:.2e} ({"Significant" if p_val < SIGNIFICANCE_LEVEL else "Not sig."})')

In [ ]:
# Score correlations
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[score_cols + ['total_score']].corr(), annot=True, cmap='coolwarm',
            center=0, fmt='.3f', ax=ax, square=True)
ax.set_title('Score Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [ ]:
# Gender differences by subject
fig, ax = plt.subplots(figsize=(10, 6))
gender_scores = df.groupby(gender_col)[score_cols].mean()
gender_scores.plot(kind='bar', ax=ax, color=['steelblue', 'coral', 'mediumseagreen'])
ax.set_title('Average Score by Gender and Subject')
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Parental education impact visualization
fig, ax = plt.subplots(figsize=(12, 6))
parent_order = df.groupby(parent_col)['avg_score'].mean().sort_values(ascending=False).index
sns.boxplot(data=df, x=parent_col, y='avg_score', order=parent_order, palette='viridis', ax=ax)
ax.set_title('Average Score by Parental Education Level')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Key Findings

1. **Test preparation courses improve scores** across all subjects — the effect is statistically significant
2. **Standard lunch (indicator of socioeconomic status) correlates with higher scores** — the largest effect size
3. **Reading and writing scores are very highly correlated** (>0.9)
4. **Math scores differ by gender** — males tend slightly higher in math, females in reading/writing
5. **Parental education positively correlates with scores** — higher education = higher scores
6. **Race/ethnicity groups show score differences** that likely reflect socioeconomic factors

## Limitations

1. **Anonymized groups:** Race/ethnicity groups (A-E) prevent real demographic analysis
2. **Limited features:** No information on school quality, tutoring, study habits
3. **Lunch as SES proxy:** Free/reduced lunch is an imperfect socioeconomic indicator
4. **Cross-sectional:** No temporal or growth data
5. **Likely synthetic:** The clean, balanced nature suggests generated data

## Recommendations / Next Steps

1. Build a score prediction model incorporating all features
2. Analyze interaction effects (e.g., does test prep benefit low-SES students more?)
3. Apply clustering to identify student archetypes

### How to Extend This Analysis
- Add real school performance data for validation
- Build an interactive dashboard for school administrators
- Apply causal inference methods to estimate test prep impact

## Common Mistakes

1. **Attributing causation to correlations:** Lunch type doesn't cause scores — it's a proxy
2. **Ignoring confounders:** Gender × test prep interactions may exist
3. **Not checking effect sizes:** Statistical significance doesn't mean practical significance
4. **Treating anonymized groups as real demographics:** Groups A-E are abstractions
5. **Over-interpreting small differences:** Some gender gaps are statistically significant but small

## Mini Challenge / Exercises

1. Which combination of factors predicts the highest average score?
2. Calculate Cohen's d for the test prep effect on math scores
3. Is the gender gap in math scores significant after controlling for test prep?
4. Create a 'risk score' for students likely to fail (score < 50 in any subject)
5. Build a simple linear regression predicting total score from all features

In [ ]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Socioeconomic factors (lunch type) have the largest impact** on student performance
- **Test preparation courses help** — a clear, actionable intervention
- **Parental education matters** but is less modifiable than direct interventions
- **Subject scores are highly correlated** — students tend to perform consistently across subjects
- **Gender differences are subject-specific** and relatively small compared to SES effects

This analysis demonstrates education policy-relevant EDA techniques and the importance of considering multiple factors simultaneously.